# Alzheimer's Disease Prediction - Data Preprocessing

This notebook cleans and prepares the OASIS Longitudinal dataset 
for machine learning.

Steps covered:
- Drop irrelevant columns
- Handle missing values
- Encode categorical variables
- Scale numerical features
- Save the cleaned dataset

Dataset: OASIS Longitudinal MRI Data

Author: Madina Alizada

## Step 1 - Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import os

print("Libraries imported successfully!")

Libraries imported successfully!


We are importing the pandas library and giving it a nickname "pd". Pandas is our main tool for working with the CSV data. It lets us load, clean and manipulate data in table format. We use "pd" as a shortcut so we do not have to type "pandas" every time.

Numpy is a math library. We give it the nickname "np". Even though we do not use it directly much, pandas and sklearn use it behind the scenes for all numerical calculations.

Sklearn is our machine learning library. We are importing two specific tools from it. LabelEncoder converts text categories like "M" and "F" into numbers like 0 and 1 because machine learning models only understand numbers, not text. StandardScaler rescales all numerical columns so they are on the same scale. For example Age ranges from 60 to 98 but nWBV ranges from 0.64 to 0.89. Without scaling, the model thinks Age is more important just because its numbers are bigger. 

OS stands for Operating System. This library lets Python talk to your computer's file system. We need it later to create folders and save our cleaned CSV file to the right location.

## Step 2 - Loading the Dataset

In [2]:
df = pd.read_csv("../data/oasis_longitudinal.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)
df.head()

Dataset loaded successfully!
Shape: (373, 15)


,Subject ID,MRI ID,Group,Visit,MR Delay,M/F,Hand,Age,EDUC,SES,MMSE,CDR,eTIV,nWBV,ASF
0,OAS2_0001,OAS2_0001_MR1,Nondemented,1,0,M,R,87,14,2.0,27.0,0.0,1987,0.696,0.883
1,OAS2_0001,OAS2_0001_MR2,Nondemented,2,457,M,R,88,14,2.0,30.0,0.0,2004,0.681,0.876
2,OAS2_0002,OAS2_0002_MR1,Demented,1,0,M,R,75,12,NaN,23.0,0.5,1678,0.736,1.046
3,OAS2_0002,OAS2_0002_MR2,Demented,2,560,M,R,76,12,NaN,28.0,0.5,1738,0.713,1.010
4,OAS2_0002,OAS2_0002_MR3,Demented,3,1895,M,R,80,12,NaN,22.0,0.5,1698,0.701,1.034


In this step we are loading our raw dataset into Python using pandas. 
The read_csv function reads the CSV file and stores it in a variable 
called df, which stands for dataframe. A dataframe is basically a 
table with rows and columns, just like Excel but inside Python.

We then print the shape which tells us how many rows and columns 
we have. Finally df.head() shows us the first 5 rows so we can 
visually confirm the data loaded correctly.

## Step 3 - Dropping Irrelevant Columns


In [3]:

columns_to_drop = ["Subject ID", "MRI ID", "Hand", "MR Delay", "CDR"]

df = df.drop(columns=columns_to_drop)

print("Columns dropped successfully!")
print("Remaining columns:", list(df.columns))
print("New shape:", df.shape)

Columns dropped successfully!
Remaining columns: ['Group', 'Visit', 'M/F', 'Age', 'EDUC', 'SES', 'MMSE', 'eTIV', 'nWBV', 'ASF']
New shape: (373, 10)


In this step we are removing columns that will not help our model 
make predictions.

Here is why we drop each one:

Subject ID - this is just a label to identify patients. 
It has no medical meaning and would confuse the model.

MRI ID - same as Subject ID, just a scan identifier. 
Not useful for prediction.

Hand - almost every single patient in this dataset is 
right-handed. A column with almost no variation teaches 
the model nothing.

MR Delay - this is the number of days between visits. 
It is a scheduling detail, not a clinical measurement.

CDR - this is the Clinical Dementia Rating. We are dropping 
it because it directly measures dementia, which is basically 
the same thing as our target variable Group. If we kept it, 
the model would just learn from CDR and not from the real 
brain and cognitive features. That would be cheating.

After dropping these 5 columns we go from 15 columns down to 10.

## Step 4 - Handling Missing Values

In [4]:

print("Missing values before:")
print(df.isnull().sum())

# Fill missing SES and MMSE with median value
df["SES"] = df["SES"].fillna(df["SES"].median())
df["MMSE"] = df["MMSE"].fillna(df["MMSE"].median())

print()
print("Missing values after:")
print(df.isnull().sum())

Missing values before:
Group     0
Visit     0
M/F       0
Age       0
EDUC      0
SES      19
MMSE      2
eTIV      0
nWBV      0
ASF       0
dtype: int64

Missing values after:
Group    0
Visit    0
M/F      0
Age      0
EDUC     0
SES      0
MMSE     0
eTIV     0
nWBV     0
ASF      0
dtype: int64


In this step we are dealing with the missing values we discovered 
during our EDA. We had 19 missing values in the SES column and 
2 missing values in the MMSE column.

We are filling them with the median value of each column. 
The median is the middle value when all numbers are sorted 
in order. We use median instead of average because the median 
is not affected by extreme outliers. For example if one patient 
had an unusual SES score it would skew the average but not 
the median.

This technique is called imputation. It is better than deleting 
the rows entirely because we would lose real patient data.

## Step 5 - Encoding Categorical Variables

In [5]:
# Encode M/F column (F = 0, M = 1)
le = LabelEncoder()
df["M/F"] = le.fit_transform(df["M/F"])

print("M/F encoding:")
print(df["M/F"].value_counts())
print()

# Encode Group column (our target variable)
df["Group"] = df["Group"].map({
    "Nondemented": 0,
    "Demented": 1,
    "Converted": 2
})

print("Group encoding:")
print(df["Group"].value_counts())
print()
print("First 5 rows after encoding:")
df.head()

M/F encoding:
M/F
0    213
1    160
Name: count, dtype: int64

Group encoding:
Group
0    190
1    146
2     37
Name: count, dtype: int64

First 5 rows after encoding:


,Group,Visit,M/F,Age,EDUC,SES,MMSE,eTIV,nWBV,ASF
0,0,1,1,87,14,2.0,27.0,1987,0.696,0.883
1,0,2,1,88,14,2.0,30.0,2004,0.681,0.876
2,1,1,1,75,12,2.0,23.0,1678,0.736,1.046
3,1,2,1,76,12,2.0,28.0,1738,0.713,1.010
4,1,3,1,80,12,2.0,22.0,1698,0.701,1.034


In this step we are converting text columns into numbers because 
machine learning models can only work with numerical data. 
They cannot understand the word "Female" or "Demented" directly.

For the M/F column we use LabelEncoder which automatically assigns 
0 to F (Female) and 1 to M (Male).

For the Group column which is our target variable, we manually 
map the values so we know exactly which number means what.
0 means Nondemented, 1 means Demented, and 2 means Converted.
We do this manually instead of using LabelEncoder so we have 
full control over which number represents which diagnosis.

## Step 6 - Feature Engineering

In [10]:
# Cell 6 -- Feature Engineering
# Load original data to access Subject ID
df_orig = pd.read_csv("../data/oasis_longitudinal.csv")

# MMSE decline
df["MMSE_decline"] = df_orig.groupby("Subject ID")["MMSE"].transform("first") - df_orig["MMSE"]

# nWBV change
df["nWBV_change"] = df_orig.groupby("Subject ID")["nWBV"].transform("first") - df_orig["nWBV"]

# Age at first visit
df["Age_first_visit"] = df_orig.groupby("Subject ID")["Age"].transform("first")

print("New features created!")
print()
print(df[["Visit", "MMSE", "MMSE_decline", "nWBV", "nWBV_change", "Age_first_visit"]].head(10))

New features created!

   Visit  MMSE  MMSE_decline   nWBV  nWBV_change  Age_first_visit
0      1  27.0           0.0  0.696        0.000               87
1      2  30.0          -3.0  0.681        0.015               87
2      1  23.0           0.0  0.736        0.000               75
3      2  28.0          -5.0  0.713        0.023               75
4      3  22.0           1.0  0.701        0.035               75
5      1  28.0           0.0  0.710        0.000               88
6      2  27.0           1.0  0.718       -0.008               88
7      1  28.0           0.0  0.712        0.000               80
8      2  29.0          -1.0  0.711        0.001               80
9      3  30.0          -2.0  0.705        0.007               80


In this step we are creating three brand new columns that did 
not exist in the original dataset. This is called feature 
engineering and it is one of the most important skills in 
data science. Instead of just giving the model a snapshot 
of each patient, we are giving it information about how 
that patient is changing over time.

We reload the original CSV here because we need the Subject ID 
column to group patients correctly. We dropped Subject ID earlier 
since it is not a useful prediction feature, but we still need 
it temporarily to calculate these changes.

MMSE_decline tells us how many points a patient's cognitive 
score has changed since their very first visit. A positive 
number means they declined. A negative number means they 
actually improved slightly.

nWBV_change tells us how much a patient's brain volume has 
changed since their first visit. A positive number means 
brain shrinkage which is a biological marker of Alzheimer's 
progression.

Age_first_visit gives the model a stable baseline age for 
each patient, which is more meaningful than the age at each 
individual visit.

## Step 7 - Feature Scaling

In [11]:
# Separate features and target
X = df.drop(columns=["Group"])
y = df["Group"]

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to dataframe
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("Scaling complete!")
print()
print("Before scaling -- Age column:")
print(df["Age"].describe())
print()
print("After scaling -- Age column:")
print(X_scaled["Age"].describe())

Scaling complete!

Before scaling -- Age column:
count    373.000000
mean      77.013405
std        7.640957
min       60.000000
25%       71.000000
50%       77.000000
75%       82.000000
max       98.000000
Name: Age, dtype: float64

After scaling -- Age column:
count    3.730000e+02
mean    -3.905128e-16
std      1.001343e+00
min     -2.229597e+00
25%     -7.880533e-01
50%     -1.756695e-03
75%      6.534905e-01
max      2.750282e+00
Name: Age, dtype: float64


In this step we are scaling all our numerical features so they 
are on the same scale.

First we separate our data into two parts. X contains all the 
input features, the columns the model will learn from. y contains 
the target variable Group, which is what the model will predict.

Then we apply StandardScaler which transforms each column so that 
its mean becomes 0 and its standard deviation becomes 1. This is 
called standardization.

To understand why this matters, look at the before and after for 
the Age column. Before scaling, Age ranges from 60 to 98. After 
scaling it ranges from around -2 to 2. Now all columns are on 
the same scale and the model treats them equally.

Without scaling, a column like eTIV which has values in the 
thousands would dominate the model simply because its numbers 
are bigger, even if it is not the most important feature.

## Step 7 - Saving the Cleaned Dataset

In [12]:
import pickle

# Save the scaled features and target
X_scaled.to_csv("../data/X_scaled.csv", index=False)
y.to_csv("../data/y.csv", index=False)

# Save the scaler object so we can use it in the web app later
with open("../model/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Cleaned data saved successfully!")
print("X_scaled.csv saved to data folder")
print("y.csv saved to data folder")
print("scaler.pkl saved to model folder")

Cleaned data saved successfully!
X_scaled.csv saved to data folder
y.csv saved to data folder
scaler.pkl saved to model folder


In this final step we are saving everything we prepared so we 
can use it in the next phase without repeating all the cleaning steps.

We save X_scaled as a CSV file containing all our cleaned and 
scaled input features. We save y as a separate CSV containing 
just the target labels.

We also save the scaler object as a pkl file. pkl stands for 
pickle which is Python's way of saving any object to disk. 
We need to save the scaler because when a user inputs data 
into our web app later, we need to scale their input using 
the exact same scaler we trained on. Otherwise the numbers 
would be on a different scale and the model would make 
wrong predictions.

In [13]:
print("Final columns in X_scaled:")
print(list(X_scaled.columns))
print()
print("Shape:", X_scaled.shape)

Final columns in X_scaled:
['Visit', 'M/F', 'Age', 'EDUC', 'SES', 'MMSE', 'eTIV', 'nWBV', 'ASF', 'MMSE_decline', 'nWBV_change', 'Age_first_visit']

Shape: (373, 12)
